# How to run the workflows

> This notebook provides minimal examples for setting up and running both workflows. For more information, alternative installation methods, and workflow configuration options and collecting additional data, see the [SPECIMEN documentation on Read the Docs](https://specimen.readthedocs.io/en/latest/).

Choose **CMPB** to curate an existing draft model (or build one with CarveMe), or **HQTB** to reconstruct a model from a high-quality template. Each example below sets up a configuration and launches one run.

All example code is in this notebook. Setup creates the YAML configuration files needed by the workflows; no companion scripts or example configs are required. Replace the example data paths with your own files before running. Paths are relative to the notebook kernel's working directory, not to the configuration file.


## 1. Install and prepare a medium

Use Python 3.10 or newer. From the repository root, install SPECIMEN into the environment used by your notebook kernel:

```bash
python -m pip install -e .
```

Install DIAMOND 2.0.4 or newer for HQTB or when building a CMPB draft with CarveMe. CarveMe also needs its `carve` command and the SCIP solver used by this workflow. HQTB needs EntrezDirect if you do not supply an NCBI mapping.

Both examples use a refineGEMs media configuration for growth testing. The cell below copies the bundled example; open `media.yaml` and choose media appropriate for your organism.


In [ ]:
from pathlib import Path

from specimen.util.set_up import download_config

# The file is written into the kernel's working directory.
download_config(filename="media.yaml", type="media")


## 2. CMPB: start from a draft model

Have a draft model in SBML format ready, using the BiGG namespace. For the smallest example, use an existing CarveMe model: no genome downloads or gap-filling databases are needed.


In [ ]:
from specimen.cmpb.workflow import run as run_cmpb

Path("cmpb_run").mkdir(exist_ok=True)
download_config(filename="cmpb.yaml", type="cmpb")


Open `cmpb.yaml` and set these entries (keep the remaining defaults):

| Entry | Example value |
| --- | --- |
| `input.modelpath` | `./data/draft.xml` |
| `input.mediapath` | `./media.yaml` |
| `general.dir` | `./cmpb_run` |
| `general.modelname` | `example_cmpb` |

Set unused `USER` entries to `null`, including `general.authorinitials`, `general.strainid` and `general.organism`, so that `modelname` is used. Leave gap-filling, ModelPolisher and BOFdat disabled for this first run.

**To build the draft with CarveMe instead:** set `input.modelpath: null`, `general.protein_fasta` to your uncompressed NCBI `*_translated_cds.faa` file, and `carveme.gram` to `grampos` or `gramneg` as appropriate. CarveMe must be installed before launching this variant.

Run the next cell after saving the configuration:


In [ ]:
run_cmpb(configpath="cmpb.yaml")


Results and logs are written under `cmpb_run/cmpb_out/`.

Alternatively, launch the same configuration from a terminal in the same working directory:

```bash
specimen cmpb run cmpb.yaml
```


## 3. HQTB: start from a high-quality template

Prepare these inputs:

- A template SBML model in the BiGG namespace and its matching annotated genome (`.gbff`). Gene identifiers in the model must match the annotation.
- Your target genome's annotation (`.gbff`), GFF (`.gff`) and full nucleotide sequence (`.fna`), all from the same assembly.
- A reference protein DIAMOND database (`.dmnd`), with an NCBI mapping if available.

Download and decompress the genome files from your chosen NCBI assemblies. See [HQTB: Collecting Data](https://specimen.readthedocs.io/en/latest/hqtb/run-pipeline.html#hqtb-collecting-data) for background on these inputs. For example, build a reference database from a folder of protein FASTA files with:

```python
from specimen.util.util import create_DIAMOND_db_from_folder

create_DIAMOND_db_from_folder(
    "./data/reference_proteins", "./data", name="reference", extension="faa"
)
```

The reference FASTA files must use identifiers compatible with your NCBI mapping. This database is separate from the template model.


In [ ]:
from specimen.hqtb.workflow import run as run_hqtb

Path("hqtb_run").mkdir(exist_ok=True)
download_config(filename="hqtb.yaml", type="hqtb-basic")


Open `hqtb.yaml` and set these entries (keep the remaining defaults):

| Entry | Example value |
| --- | --- |
| `subject.annotated_genome` | `./data/target.gbff` |
| `subject.gff` | `./data/target.gff` |
| `subject.full_sequence` | `./data/target.fna` |
| `template.annotated_genome` | `./data/template.gbff` |
| `template.model` | `./data/template.xml` |
| `general.dir` | `./hqtb_run` |
| `general.modelname` | `example_hqtb` |
| `data.diamond` | `./data/reference.dmnd` |
| `parameters.general.email` | Your email address for NCBI queries |
| `parameters.analysis.media_analysis` | `./media.yaml` |

Set unused `USER` entries to `null`, including the three naming entries under `general`. If you have an NCBI mapping, set `data.ncbi_map` to its path; otherwise leave it `null`. Leave the optional gap-filling inputs disabled for this first run.

Run the next cell after saving the configuration:


In [ ]:
run_hqtb(config_file="hqtb.yaml")


Results are written into the numbered step directories under `hqtb_run/`, with progress in `hqtb_run/logs/`.

Alternatively, launch the same configuration from a terminal in the same working directory:

```bash
specimen hqtb run hqtb.yaml
```

These examples show setup and launch with your own data. A successful full run also depends on the input data and the installed external tools; HQTB is currently flagged in the repository README as potentially affected by refineGEMs refactoring.
